# RAG From Scratch — Combined Notebook (Parts 1–18)

This notebook combines all five RAG-from-scratch notebooks from the [LangChain repo](https://github.com/langchain-ai/rag-from-scratch) into a single runnable file, updated for current LangChain (≥ 0.3) / Pydantic v2.

**Table of Contents**
- [Setup](#setup)
- [Part 1 — RAG Overview](#part-1)
- [Part 2 — Indexing](#part-2)
- [Part 3 — Retrieval](#part-3)
- [Part 4 — Generation](#part-4)
- [Part 5 — Multi-Query](#part-5)
- [Part 6 — RAG-Fusion](#part-6)
- [Part 7 — Decomposition](#part-7)
- [Part 8 — Step Back](#part-8)
- [Part 9 — HyDE](#part-9)
- [Part 10 — Logical & Semantic Routing](#part-10)
- [Part 11 — Query Structuring for Metadata Filters](#part-11)
- [Part 12 — Multi-Representation Indexing](#part-12)
- [Part 13 — RAPTOR (reference)](#part-13)
- [Part 14 — ColBERT](#part-14)
- [Part 15 — Re-ranking](#part-15)
- [Parts 16-18 — CRAG, Self-RAG, Long Context (references)](#part-16-18)


---
<a id='setup'></a>
## Setup

In [3]:
# Final attempt at a clean install for the missing retrievers module
!pip uninstall -y langchain langchain-community langchain-core
!pip install -qU --no-cache-dir langchain==0.3.0 langchain-community==0.3.0 langchain-openai langchain-core==0.3.0 chromadb tiktoken bs4 pydantic numpy

Found existing installation: langchain 0.3.0
Uninstalling langchain-0.3.0:
  Successfully uninstalled langchain-0.3.0
Found existing installation: langchain-community 0.3.0
Uninstalling langchain-community-0.3.0:
  Successfully uninstalled langchain-community-0.3.0
Found existing installation: langchain-core 0.3.0
Uninstalling langchain-core-0.3.0:
  Successfully uninstalled langchain-core-0.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-sdk 0.4.7 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.0 which is incompatible.
langgraph-prebuilt 1.1.1 requires langchain-core>=1.3.1, but you have langchain-core 0.3.0 which is incompatible.
langgraph 1.2.15 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.0 which is incompatible.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 

In [1]:
import os
from getpass import getpass

# OpenAI — required for all parts
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

# LangSmith — optional but recommended for tracing
enable_tracing = True  # set to True to enable LangSmith tracing
if enable_tracing:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
    if not os.environ.get("LANGCHAIN_API_KEY"):
        os.environ["LANGCHAIN_API_KEY"] = getpass("Enter your LangSmith API key: ")

In [2]:
# Re-loading the documents and generating summaries for Part 12
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Load the blog posts again
loader1 = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
loader2 = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
multi_docs = loader1.load() + loader2.load()

# Summarize each document
summary_chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}")
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
)
summaries = summary_chain.batch(multi_docs, {"max_concurrency": 5})
print(f"Re-loaded {len(multi_docs)} docs and generated {len(summaries)} summaries.")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Re-loaded 2 docs and generated 2 summaries.


In [3]:
import uuid
from langchain_core.stores import InMemoryByteStore
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.documents import Document

# Initialize vectorstore and store
vectorstore_mv = Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings())
store = InMemoryByteStore()
id_key = "doc_id"
retriever_mv = MultiVectorRetriever(vectorstore=vectorstore_mv, byte_store=store, id_key=id_key)

# Map summaries to documents
doc_ids = [str(uuid.uuid4()) for _ in multi_docs]
summary_docs = [Document(page_content=s, metadata={id_key: doc_ids[i]}) for i, s in enumerate(summaries)]

# Add to retriever
retriever_mv.vectorstore.add_documents(summary_docs)
retriever_mv.docstore.mset(list(zip(doc_ids, multi_docs)))
print("MultiVectorRetriever is now ready.")

/var/folders/88/87fxhpyj0l7fzlf3yjrjpz1m0000gn/T/ipykernel_92916/2079773116.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  vectorstore_mv = Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings())


MultiVectorRetriever is now ready.


---
<a id='part-1'></a>
## Part 1 — RAG Overview

A minimal end-to-end RAG pipeline: **Index → Retrieve → Generate**.

Reference: [RAG quickstart](https://python.langchain.com/docs/use_cases/question_answering/quickstart)

In [4]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# ── INDEXING ────────────────────────────────────────────────────────────────

# Load
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Embed + store
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

# ── RETRIEVAL + GENERATION ──────────────────────────────────────────────────

# Standard RAG prompt (equivalent to hub.pull("rlm/rag-prompt"))
rag_prompt = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "Question: {question}\n\n"
    "Context: {context}\n\n"
    "Answer:"
)

# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")

'Task Decomposition is the process of breaking down a complex task into smaller, manageable steps to facilitate planning and execution. Techniques like Chain of Thought (CoT) and Tree of Thoughts enhance this process by encouraging step-by-step reasoning and exploring multiple possibilities. It can be achieved through simple prompts, task-specific instructions, or human inputs, and may also involve external planners for long-horizon tasks.'

---
<a id='part-2'></a>
## Part 2 — Indexing

Deeper look at the three indexing components: **tokens**, **embeddings**, and **vector stores**.

In [18]:
import tiktoken

question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Return the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    return len(encoding.encode(string))

print("Tokens in question:", num_tokens_from_string(question, "cl100k_base"))

Tokens in question: 8


In [19]:
import numpy as np
from langchain_openai import OpenAIEmbeddings

embd = OpenAIEmbeddings()
query_result = embd.embed_query(question)
document_result = embd.embed_query(document)

print("Embedding dimension:", len(query_result))

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    return dot_product / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

Embedding dimension: 1536
Cosine Similarity: 0.8806915835035412


In [20]:
# Load blog
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

# Split using tiktoken-aware splitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, chunk_overlap=50
)
splits = text_splitter.split_documents(blog_docs)
print(f"Number of chunks: {len(splits)}")

# Index
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()
print("Vectorstore and retriever ready.")

Number of chunks: 50
Vectorstore and retriever ready.


---
<a id='part-3'></a>
## Part 3 — Retrieval

Using the vectorstore retriever with `k=1` to fetch the single most relevant chunk.

In [21]:
# Retriever with k=1 (reuses vectorstore from Part 2)
retriever_k1 = vectorstore.as_retriever(search_kwargs={"k": 1})

docs_retrieved = retriever_k1.invoke("What is Task Decomposition?")
print(f"Retrieved {len(docs_retrieved)} doc(s)")
print(docs_retrieved[0].page_content[:300])

Retrieved 1 doc(s)
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructe


---
<a id='part-4'></a>
## Part 4 — Generation

Build a custom prompt and compose the full RAG chain with LCEL.

In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Custom prompt template
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Simple chain (pass docs directly)
chain = prompt | llm

result = chain.invoke({"context": docs_retrieved, "question": "What is Task Decomposition?"})
print(result.content)

Task Decomposition is the process of breaking down a complicated task into smaller and simpler steps. It involves techniques like Chain of Thought (CoT), which encourages a model to think step by step, and Tree of Thoughts, which explores multiple reasoning possibilities at each step by creating a tree structure of thoughts. Task decomposition can be achieved through various methods, including simple prompting, task-specific instructions, or human inputs.


In [23]:
from langchain_core.prompts import ChatPromptTemplate

# Standard RAG prompt (replaces hub.pull("rlm/rag-prompt"))
# This is the exact template used in the LangChain Hub version.
RAG_PROMPT_TEMPLATE = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "Question: {question}\n\n"
    "Context: {context}\n\n"
    "Answer:"
)

prompt_hub_rag = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)
print(prompt_hub_rag.messages[0].prompt.template)

You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

Question: {question}

Context: {context}

Answer:


In [24]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Full RAG chain using the custom prompt and the k=1 retriever
rag_chain = (
    {"context": retriever_k1, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")

'Task Decomposition is the process of breaking down a complicated task into smaller and simpler steps. It involves techniques like Chain of Thought (CoT), which encourages a model to think step by step, and Tree of Thoughts, which explores multiple reasoning possibilities at each step by creating a tree structure of thoughts. Task decomposition can be achieved through simple prompting, task-specific instructions, or human inputs.'

---
<a id='part-5'></a>
## Part 5 — Multi-Query

Generate **multiple rephrased versions** of the query and take the union of retrieved docs to overcome limitations of distance-based search.

Docs: https://reference.langchain.com/python/langchain-classic/retrievers/multi_query/MultiQueryRetriever


In [25]:
# Re-build shared index used by Parts 5-9 and 15
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, chunk_overlap=50
)
splits = text_splitter.split_documents(blog_docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()
print("Shared index ready for Parts 5-9 and 15.")

Shared index ready for Parts 5-9 and 15.


In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Prompt: generate 5 alternative questions
template = """You are an AI language model assistant. Your task is to generate five
different versions of the given user question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search.
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_perspectives
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

In [27]:
import json
from langchain_core.documents.base import Document

def get_unique_union(documents: list[list]):
    """Return unique union of retrieved docs across multiple query results."""
    # Using json.dumps to serialize Document objects to strings for uniqueness
    flattened_docs = [json.dumps(doc.dict()) for sublist in documents for doc in sublist]
    unique_docs_str = list(set(flattened_docs))
    # Using Document.parse_raw to deserialize strings back to Document objects
    return [Document.parse_raw(doc_str) for doc_str in unique_docs_str]

In [29]:
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Fix: Define the retrieval_chain that was missing in this cell
# This combines query generation, retrieval, and document deduplication
retrieval_chain = generate_queries | retriever.map() | get_unique_union

template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

final_rag_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

# Use the specific question for this part
question_part5 = "What is task decomposition for LLM agents?"
final_rag_chain.invoke({"question": question_part5})

/tmp/ipykernel_14762/2490539397.py:7: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  flattened_docs = [json.dumps(doc.dict()) for sublist in documents for doc in sublist]
/tmp/ipykernel_14762/2490539397.py:10: PydanticDeprecatedSince20: The `parse_raw` method is deprecated; if your data is JSON use `model_validate_json`, otherwise load the data then use `model_validate` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return [Document.parse_raw(doc_str) for doc_str in unique_docs_str]


'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable sub-tasks or steps. This can be achieved through various methods:\n\n1. **Simple Prompting**: LLMs can be prompted with questions or instructions such as "Steps for XYZ." or "What are the subgoals for achieving XYZ?" to elicit a breakdown of the task.\n\n2. **Task-Specific Instructions**: Providing specific instructions tailored to the task at hand, such as "Write a story outline." for writing a novel, can guide the LLM in decomposing the task effectively.\n\n3. **Human Inputs**: Involving human input can also facilitate task decomposition, allowing for more nuanced and context-aware breakdowns.\n\nAdditionally, techniques like Chain of Thought (CoT) and Tree of Thoughts enhance this process. CoT encourages the model to "think step by step," while Tree of Thoughts explores multiple reasoning possibilities at each step, creating a tree structure of

In [30]:
question = "What is task decomposition for LLM agents?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs_multi = retrieval_chain.invoke({"question": question})
print(f"Unique docs retrieved: {len(docs_multi)}")

Unique docs retrieved: 3


/tmp/ipykernel_14762/2490539397.py:7: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  flattened_docs = [json.dumps(doc.dict()) for sublist in documents for doc in sublist]
/tmp/ipykernel_14762/2490539397.py:10: PydanticDeprecatedSince20: The `parse_raw` method is deprecated; if your data is JSON use `model_validate_json`, otherwise load the data then use `model_validate` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return [Document.parse_raw(doc_str) for doc_str in unique_docs_str]


In [31]:
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

final_rag_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question": question})

/tmp/ipykernel_14762/2490539397.py:7: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  flattened_docs = [json.dumps(doc.dict()) for sublist in documents for doc in sublist]
/tmp/ipykernel_14762/2490539397.py:10: PydanticDeprecatedSince20: The `parse_raw` method is deprecated; if your data is JSON use `model_validate_json`, otherwise load the data then use `model_validate` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return [Document.parse_raw(doc_str) for doc_str in unique_docs_str]


'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable sub-tasks or steps. This can be achieved through various methods:\n\n1. **Simple Prompting**: LLMs can be prompted with questions or instructions such as "Steps for XYZ." or "What are the subgoals for achieving XYZ?" to elicit a breakdown of the task.\n\n2. **Task-Specific Instructions**: Providing specific instructions tailored to the task at hand, such as "Write a story outline." for writing a novel, can guide the LLM in decomposing the task effectively.\n\n3. **Human Inputs**: Involving human input can also facilitate task decomposition, allowing for more nuanced and context-aware breakdowns.\n\nAdditionally, techniques like Chain of Thought (CoT) and Tree of Thoughts (ToT) enhance this process. CoT encourages the model to think step by step, while ToT explores multiple reasoning possibilities at each step, creating a tree structure of thoughts

---
<a id='part-6'></a>
## Part 6 — RAG-Fusion

Generate multiple related queries, retrieve for each, then **re-rank** all results using **Reciprocal Rank Fusion (RRF)**.

Blog: https://towardsdatascience.com/forget-rag-the-future-is-rag-fusion-1147298d8ad1

In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

template = """You are a helpful assistant that generates multiple search queries based on a single input query.
Generate multiple search queries related to: {question}
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

generate_queries_fusion = (
    prompt_rag_fusion
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

In [33]:
import json
from langchain_core.documents.base import Document

def reciprocal_rank_fusion(results: list[list], k: int = 60):
    """Re-rank results from multiple retrieval lists using RRF."""
    fused_scores: dict = {}
    for docs in results:
        for rank, doc in enumerate(docs):
            # Use json.dumps and doc.dict() for serialization
            doc_str = json.dumps(doc.model_dump()) # Updated: doc.dict() to doc.model_dump()
            fused_scores[doc_str] = fused_scores.get(doc_str, 0) + 1 / (rank + k)
    return [
        # Use Document.parse_raw for deserialization
        (Document.model_validate_json(doc_str), score) # Updated: Document.parse_raw() to Document.model_validate_json()
        for doc_str, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

question = "What is task decomposition for LLM agents?"
retrieval_chain_rag_fusion = generate_queries_fusion | retriever.map() | reciprocal_rank_fusion
docs_fused = retrieval_chain_rag_fusion.invoke({"question": question})
print(f"Re-ranked docs: {len(docs_fused)}")

Re-ranked docs: 6


In [34]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

final_rag_chain_fusion = (
    {"context": retrieval_chain_rag_fusion, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain_fusion.invoke({"question": question})

'Task decomposition for LLM (Large Language Model) agents involves breaking down complex tasks into smaller, more manageable steps to facilitate planning and execution. This can be achieved through various methods:\n\n1. **Prompting**: Simple prompts can be used to guide the LLM in identifying steps or subgoals for a task. For example, prompts like "Steps for XYZ" or "What are the subgoals for achieving XYZ?" can help in outlining the necessary actions.\n\n2. **Task-Specific Instructions**: Providing specific instructions tailored to the task at hand can also aid in decomposition. For instance, asking the LLM to "Write a story outline" for a novel directs it to focus on the relevant components of that task.\n\n3. **Human Inputs**: Involving human guidance can further enhance the decomposition process, allowing for more nuanced and context-aware breakdowns of tasks.\n\nAdditionally, advanced techniques like Chain of Thought (CoT) and Tree of Thoughts (ToT) can be employed. CoT encourage

---
<a id='part-7'></a>
## Part 7 — Decomposition

Break a complex question into **sub-questions** and answer them in two ways:
1. **Recursively** — each answer informs the next  
2. **Individually** — answer each in parallel, then synthesize

Papers:
- https://arxiv.org/pdf/2205.10625.pdf
- https://arxiv.org/abs/2212.10509.pdf

In [35]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

template = """You are a helpful assistant that generates multiple sub-questions related to an input question.
The goal is to break down the input into a set of sub-problems / sub-questions that can be answered in isolation.
Generate multiple search queries related to: {question}
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

generate_queries_decomposition = (
    prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

question = "What are the main components of an LLM-powered autonomous agent system?"
sub_questions = generate_queries_decomposition.invoke({"question": question})
print("\n".join(sub_questions))

1. What are the key components of a large language model (LLM) used in autonomous agent systems?
2. How do perception and sensor integration work in LLM-powered autonomous agents?
3. What role does decision-making and planning play in the functionality of LLM-based autonomous systems?


In [36]:
# ── Answer Recursively ────────────────────────────────────────────────────────
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents.base import Document # Added import for Document for type hinting

decomposition_prompt = ChatPromptTemplate.from_template("""
Here is the question you need to answer:

--- {question} ---

Here is any available background question + answer pairs:

--- {q_a_pairs} ---

Here is additional context relevant to the question:

--- {context} ---

Use the above context and any background Q&A pairs to answer: {question}
""")

def format_qa_pair(question, answer):
    return f"Question: {question}\nAnswer: {answer}".strip()

def format_docs(docs: list[Document]) -> str:
    """Format a list of Document objects into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

q_a_pairs = ""
answer = "" # Initialize answer to avoid UnboundLocalError if loop doesn't run
for q in sub_questions:
    if not q.strip():
        continue
    rag_chain = (
        {
            "context": itemgetter("question") | retriever | format_docs, # Added format_docs
            "question": itemgetter("question"),
            "q_a_pairs": itemgetter("q_a_pairs"),
        }
        | decomposition_prompt
        | llm
        | StrOutputParser()
    )
    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})
    q_a_pairs += "\n---\n" + format_qa_pair(q, answer)

print("Final answer (recursive decomposition):")
print(answer)

Final answer (recursive decomposition):
Decision-making and planning are critical components in the functionality of LLM-based autonomous systems, serving as the backbone for effective task execution and adaptive behavior. Here are the key roles they play:

1. **Task Decomposition and Subgoal Identification**:
   - LLMs enable autonomous agents to break down complex tasks into smaller, manageable subgoals. This decomposition is essential for structuring problems in a way that makes them easier to tackle. By identifying subgoals, the agent can focus on achieving these smaller objectives sequentially, which enhances efficiency and clarity in task execution.

2. **Dynamic Planning**:
   - LLMs facilitate dynamic planning by allowing agents to adapt their strategies based on real-time feedback from their environment. As agents gather sensory data, they can reflect on their actions and adjust their plans accordingly. This iterative process of planning and reflection ensures that the agent c

In [37]:
# ── Answer Individually ───────────────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Inline RAG prompt (replaces hub.pull("rlm/rag-prompt"))
prompt_rag = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "Question: {question}\n\n"
    "Context: {context}\n\n"
    "Answer:"
)

def retrieve_and_rag(question, prompt_rag, sub_question_generator_chain):
    """Run RAG on each generated sub-question."""
    sub_questions = sub_question_generator_chain.invoke({"question": question})
    rag_results = []
    for sub_question in sub_questions:
        if not sub_question.strip():
            continue
        retrieved_docs = retriever.invoke(sub_question)
        answer = (prompt_rag | llm | StrOutputParser()).invoke(
            {"context": retrieved_docs, "question": sub_question}
        )
        rag_results.append(answer)
    return rag_results, sub_questions

answers, questions_list = retrieve_and_rag(question, prompt_rag, generate_queries_decomposition)

def format_qa_pairs(questions, answers):
    return "\n\n".join(
        f"Question {i}: {q}\nAnswer {i}: {a}"
        for i, (q, a) in enumerate(zip(questions, answers), 1)
    ).strip()

context = format_qa_pairs(questions_list, answers)

# Synthesize final answer
synthesis_prompt = ChatPromptTemplate.from_template("""
Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
""")

final_rag_chain = synthesis_prompt | llm | StrOutputParser()
final_rag_chain.invoke({"context": context, "question": question})

"An LLM-powered autonomous agent system comprises several main components that work together to enhance its functionality. Key components include:\n\n1. **Large Language Model (LLM)**: This serves as the core of the system, acting as the agent's brain and facilitating various cognitive processes such as planning, decision-making, and subgoal decomposition.\n\n2. **Planning and Subgoal Decomposition**: The agent is capable of breaking down complex tasks into smaller, manageable subgoals, which allows for more efficient task handling and organization.\n\n3. **Self-Reflection and Learning**: The system incorporates mechanisms for self-critique, enabling the agent to learn from past actions and refine its strategies for future tasks, thereby improving its performance over time.\n\n4. **Memory**: Memory plays a crucial role in enhancing the agent's capabilities, allowing it to retain information from past experiences and utilize it in decision-making processes.\n\n5. **Perception and Sensor

---
<a id='part-8'></a>
## Part 8 — Step Back

Paraphrase the question into a **more generic step-back question** to retrieve broader context before answering the specific question.

Paper: https://arxiv.org/pdf/2310.06117.pdf

In [38]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Few-shot examples
examples = [
    {"input": "Could the members of The Police perform lawful arrests?",
     "output": "what can the members of The Police do?"},
    {"input": "Jan Sindel's was born in what country?",
     "output": "what is Jan Sindel's personal history?"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [("human", "{input}"), ("ai", "{output}")]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt, examples=examples
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert at world knowledge. Your task is to step back and "
                   "paraphrase a question to a more generic step-back question, which is "
                   "easier to answer. Here are a few examples:"),
        few_shot_prompt,
        ("user", "{question}"),
    ]
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
generate_queries_step_back = prompt | llm | StrOutputParser()

question = "What is task decomposition for LLM agents?"
step_back_q = generate_queries_step_back.invoke({"question": question})
print("Step-back question:", step_back_q)

Step-back question: what does task decomposition mean in the context of agents?


In [39]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question.
Your response should be comprehensive and not contradict the following context if it is relevant.
Otherwise, ignore it if not relevant.

Normal context:
{normal_context}

Step-back context:
{step_back_context}

Original Question: {question}
Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

chain = (
    {
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        "step_back_context": generate_queries_step_back | retriever,
        "question": lambda x: x["question"],
    }
    | response_prompt
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
)

chain.invoke({"question": question})

'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable sub-tasks or steps. This is essential for effective planning and execution, as complicated tasks often involve multiple steps that need to be addressed sequentially or in parallel.\n\nThere are several methods for task decomposition:\n\n1. **Prompting Techniques**: LLMs can be prompted with simple instructions to elicit a breakdown of tasks. For example, prompts like "Steps for XYZ.\\\\n1." or "What are the subgoals for achieving XYZ?" can guide the model to generate a list of steps or subgoals.\n\n2. **Task-Specific Instructions**: Providing specific instructions tailored to the task can also facilitate decomposition. For instance, asking the model to "Write a story outline." directs it to focus on the particular requirements of story writing.\n\n3. **Human Inputs**: Human guidance can be utilized to inform the LLM about the necessary steps or co

---
<a id='part-9'></a>
## Part 9 — HyDE (Hypothetical Document Embeddings)

Generate a **hypothetical answer** to the question, embed it, and use it to retrieve semantically similar real documents.

Paper: https://arxiv.org/abs/2212.10496

In [40]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Generate a hypothetical document
template = """Please write a scientific paper passage to answer the question.
Question: {question}
Passage:"""
prompt_hyde = ChatPromptTemplate.from_template(template)

generate_docs_for_retrieval = (
    prompt_hyde | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser()
)

question = "What is task decomposition for LLM agents?"
hypothetical_doc = generate_docs_for_retrieval.invoke({"question": question})
print(hypothetical_doc[:400])

**Task Decomposition for LLM Agents**

Task decomposition is a critical process in the context of Large Language Model (LLM) agents, enabling them to effectively tackle complex problems by breaking them down into smaller, more manageable sub-tasks. This approach is grounded in cognitive science and artificial intelligence, where the ability to simplify intricate tasks enhances both efficiency and 


In [41]:
# Retrieve using the hypothetical document, then answer with real context
retrieval_chain = generate_docs_for_retrieval | retriever
retrieved_docs = retrieval_chain.invoke({"question": question})

template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

final_rag_chain = prompt | llm | StrOutputParser()
final_rag_chain.invoke({"context": retrieved_docs, "question": question})

'Task decomposition for LLM (Large Language Model) agents involves breaking down complex tasks into smaller, more manageable steps to facilitate planning and execution. This can be achieved through various methods:\n\n1. **Chain of Thought (CoT)**: This technique encourages the model to "think step by step," allowing it to utilize more computational resources at test time to decompose difficult tasks into simpler components. CoT helps in transforming large tasks into multiple manageable tasks and provides insight into the model\'s reasoning process.\n\n2. **Tree of Thoughts**: This approach extends CoT by exploring multiple reasoning possibilities at each step. It decomposes the problem into several thought steps and generates multiple thoughts for each step, creating a tree structure. The search process can be conducted using breadth-first search (BFS) or depth-first search (DFS), with each state evaluated by a classifier or through majority voting.\n\n3. **Prompting Techniques**: Tas

---
<a id='part-10'></a>
## Part 10 — Logical & Semantic Routing

Route a query to the correct data source or prompt using **structured output** (logical routing) or **embedding similarity** (semantic routing).

Docs: https://python.langchain.com/docs/use_cases/query_analysis/techniques/routing

In [42]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Data model for routing decision
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""
    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question choose which datasource is most relevant.",
    )

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm = llm.with_structured_output(RouteQuery)

system = """You are an expert at routing a user question to the appropriate data source.
Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("human", "{question}")]
)
router = prompt | structured_llm

question = """
Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""
result = router.invoke({"question": question})
print("Routed to:", result.datasource)

Routed to: python_docs


In [43]:
from langchain_core.runnables import RunnableLambda

def choose_route(result):
    if "python_docs" in result.datasource.lower():
        return "chain for python_docs"
    elif "js_docs" in result.datasource.lower():
        return "chain for js_docs"
    else:
        return "chain for golang_docs"

full_chain = router | RunnableLambda(choose_route)
full_chain.invoke({"question": question})

'chain for python_docs'

In [44]:
import numpy as np
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

embeddings = OpenAIEmbeddings()
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)

def prompt_router(input):
    query_embedding = embeddings.embed_query(input["query"])
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    most_similar = prompt_templates[similarity.argmax()]
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)

chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

print(chain.invoke("What's a black hole"))

Using PHYSICS
A black hole is a region in space where the gravitational pull is so strong that nothing, not even light, can escape from it. This occurs when a massive star collapses under its own gravity at the end of its life cycle. The boundary surrounding a black hole is called the event horizon; once something crosses this boundary, it cannot return. Black holes can vary in size, from stellar black holes (formed from collapsing stars) to supermassive black holes (found at the centers of galaxies, containing millions or even billions of times the mass of the Sun). Their properties are described by Einstein's theory of general relativity.


---
<a id='part-11'></a>
## Part 11 — Query Structuring for Metadata Filters

Convert a natural language question into a **structured query** with metadata filters (date ranges, view counts, etc.) that can be applied to a vectorstore.

Docs: https://python.langchain.com/docs/use_cases/query_analysis/techniques/structuring

In [45]:
# 1. Install/Update dependencies
#!pip install -qU pytube youtube-transcript-api

import datetime
from langchain_community.document_loaders import YoutubeLoader
from langchain_core.documents import Document

try:
    # Main Attempt: Standard loader
    loader = YoutubeLoader.from_youtube_url(
        "https://www.youtube.com/watch?v=pbAd8O1Lvm4",
        add_video_info=False
    )
    docs = loader.load()
except Exception as e:
    print(f"Standard loader failed ({e}). Attempting direct transcript fetch...")
    try:
        from youtube_transcript_api import YouTubeTranscriptApi
        # Direct API call to bypass potential shadowing in LangChain's loader
        video_id = "pbAd8O1Lvm4"
        transcript = YouTubeTranscriptApi.get_transcript(video_id)
        text = " ".join([t["text"] for t in transcript])
        docs = [Document(page_content=text, metadata={"source": video_id})]
    except Exception as e2:
        print(f"Direct fetch failed ({e2}). Providing placeholder transcript.")
        # Absolute fallback to ensure notebook flow isn't broken
        docs = [Document(page_content="This is a placeholder transcript about RAG and LangGraph.", metadata={"source": "fallback"})]

# Populate metadata for the Query Analyzer section
if docs:
    docs[0].metadata.update({
        "title": "Self-reflective RAG with LangGraph: Self-RAG and CRAG",
        "view_count": 12000,
        "publish_date": "2024-02-07 00:00:00",
        "length": 1058,
        "author": "LangChain",
    })
    print("Document loaded successfully with metadata:", docs[0].metadata)

Standard loader failed (type object 'YouTubeTranscriptApi' has no attribute 'list_transcripts'). Attempting direct transcript fetch...
Direct fetch failed (type object 'YouTubeTranscriptApi' has no attribute 'get_transcript'). Providing placeholder transcript.
Document loaded successfully with metadata: {'source': 'fallback', 'title': 'Self-reflective RAG with LangGraph: Self-RAG and CRAG', 'view_count': 12000, 'publish_date': '2024-02-07 00:00:00', 'length': 1058, 'author': 'LangChain'}


In [46]:
import datetime
from typing import Literal, Optional
from pydantic import BaseModel, Field

class TutorialSearch(BaseModel):
    """Search over a database of tutorial videos about a software library."""

    content_search: str = Field(
        ..., description="Similarity search query applied to video transcripts."
    )
    title_search: str = Field(
        ...,
        description=(
            "Alternate version of the content search query to apply to video titles. "
            "Should be succinct and only include key words that could be in a video title."
        ),
    )
    min_view_count: Optional[int] = Field(
        None, description="Minimum view count filter, inclusive. Only use if explicitly specified."
    )
    max_view_count: Optional[int] = Field(
        None, description="Maximum view count filter, exclusive. Only use if explicitly specified."
    )
    earliest_publish_date: Optional[datetime.date] = Field(
        None, description="Earliest publish date filter, inclusive. Only use if explicitly specified."
    )
    latest_publish_date: Optional[datetime.date] = Field(
        None, description="Latest publish date filter, exclusive. Only use if explicitly specified."
    )
    min_length_sec: Optional[int] = Field(
        None, description="Minimum video length in seconds, inclusive. Only use if explicitly specified."
    )
    max_length_sec: Optional[int] = Field(
        None, description="Maximum video length in seconds, exclusive. Only use if explicitly specified."
    )

    def pretty_print(self) -> None:
        for field_name in self.model_fields:
            value = getattr(self, field_name)
            if value is not None:
                print(f"{field_name}: {value}")

In [47]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system = """You are an expert at converting user questions into database queries. \
You have access to a database of tutorial videos about a software library for building LLM-powered applications. \
Given a question, return a database query optimized to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, do not try to rephrase them."""

prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("human", "{question}")]
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm = llm.with_structured_output(TutorialSearch)
query_analyzer = prompt | structured_llm

In [48]:
print("--- Query 1: 'rag from scratch' ---")
query_analyzer.invoke({"question": "rag from scratch"}).pretty_print()

print("\n--- Query 2: 'videos on chat langchain published in 2023' ---")
query_analyzer.invoke({"question": "videos on chat langchain published in 2023"}).pretty_print()

print("\n--- Query 3: 'how to use multi-modal models in an agent, only videos under 5 minutes' ---")
query_analyzer.invoke(
    {"question": "how to use multi-modal models in an agent, only videos under 5 minutes"}
).pretty_print()

--- Query 1: 'rag from scratch' ---


/tmp/ipykernel_14762/527750133.py:38: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  for field_name in self.model_fields:


content_search: rag from scratch
title_search: rag from scratch

--- Query 2: 'videos on chat langchain published in 2023' ---


/tmp/ipykernel_14762/527750133.py:38: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  for field_name in self.model_fields:


content_search: chat langchain
title_search: chat langchain
earliest_publish_date: 2023-01-01
latest_publish_date: 2023-12-31

--- Query 3: 'how to use multi-modal models in an agent, only videos under 5 minutes' ---
content_search: multi-modal models in an agent
title_search: multi-modal models agent
max_length_sec: 300


/tmp/ipykernel_14762/527750133.py:38: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  for field_name in self.model_fields:


---
<a id='part-12'></a>
## Part 12 — Multi-Representation Indexing

Index **summaries** for retrieval while storing **full documents** for generation. Uses `MultiVectorRetriever`.

Docs: https://python.langchain.com/docs/modules/data_connection/retrievers/multi_vector  
Paper: https://arxiv.org/abs/2312.06648

In [49]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Load two blog posts
loader1 = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
loader2 = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
multi_docs = loader1.load() + loader2.load()
print(f"Loaded {len(multi_docs)} documents")

# Summarise each document
chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}")
    | ChatOpenAI(model="gpt-4o-mini", max_retries=0)
    | StrOutputParser()
)
summaries = chain.batch(multi_docs, {"max_concurrency": 5})
print("Summaries generated:", len(summaries))

Loaded 2 documents


Summaries generated: 2


In [50]:
import sys
import uuid
import importlib
import os

# Forcefully clear modules from sys.modules to resolve MRO conflicts
for mod in list(sys.modules):
    if mod.startswith("langchain") or mod.startswith("pydantic"):
        del sys.modules[mod]

# Re-import core components
import langchain_core
import langchain
import pydantic
from langchain_core.stores import InMemoryByteStore
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Robust import for MultiVectorRetriever
try:
    from langchain.retrievers.multi_vector import MultiVectorRetriever
except (ImportError, ModuleNotFoundError):
    from langchain.retrievers import MultiVectorRetriever

# Ensure API Key is set
if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY not found.")

# Initialize vectorstore and store
vectorstore_mv = Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings())
store = InMemoryByteStore()
id_key = "doc_id"
retriever_mv = MultiVectorRetriever(vectorstore=vectorstore_mv, byte_store=store, id_key=id_key)

try:
    if 'multi_docs' not in globals() or 'summaries' not in globals():
        raise NameError("Please run Part 12 - Load & Summarize first.")

    # FIX: Cast multi_docs to the NEW Document class to avoid MRO/Type errors
    recast_multi_docs = [Document(page_content=d.page_content, metadata=d.metadata) for d in multi_docs]

    doc_ids = [str(uuid.uuid4()) for _ in recast_multi_docs]
    summary_docs = [Document(page_content=s, metadata={id_key: doc_ids[i]}) for i, s in enumerate(summaries)]

    retriever_mv.vectorstore.add_documents(summary_docs)
    retriever_mv.docstore.mset(list(zip(doc_ids, recast_multi_docs)))
    print("MultiVectorRetriever successfully initialized with recast documents.")
except Exception as e:
    print(f"Error: {e}")

/tmp/ipykernel_14762/875344368.py:31: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  vectorstore_mv = Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings())


MultiVectorRetriever successfully initialized with recast documents.


In [51]:
query = "Memory in agents"

# Sub-docs: returns the matched summary
sub_docs = vectorstore_mv.similarity_search(query, k=1)
print("Matched summary:\n", sub_docs[0].page_content[:300])

print("\n" + "="*60 + "\n")

# Full docs: retriever fetches the original full document via doc_id
retrieved_full = retriever_mv.invoke(query)
print("Full document preview (first 500 chars):\n", retrieved_full[0].page_content[:500])

Matched summary:
 The document "LLM Powered Autonomous Agents" by Lilian Weng discusses the development and implementation of autonomous agents powered by large language models (LLMs). It provides a comprehensive overview of the architecture, components, challenges, and case studies related to these agents.

### Key 


Full document preview (first 500 chars):
 





LLM Powered Autonomous Agents | Lil'Log







































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three:


---
<a id='part-13'></a>
## Part 13 — RAPTOR *(reference)*

RAPTOR recursively embeds, clusters, and summarises chunks of text, building a tree structure that supports retrieval at multiple levels of abstraction.

- Deep dive video: https://www.youtube.com/watch?v=jbGchdTL7d0  
- Paper: https://arxiv.org/pdf/2401.18059.pdf  
- Full notebook: https://github.com/langchain-ai/langchain/blob/master/cookbook/RAPTOR.ipynb

---
<a id='part-14'></a>
## Part 14 — ColBERT via RAGatouille

ColBERT generates **per-token contextual embeddings** for both queries and documents, scoring each document as the sum of max-similarity per query token.

References:
- https://hackernoon.com/how-colbert-helps-developers-overcome-the-limits-of-rag  
- https://python.langchain.com/docs/integrations/retrievers/ragatouille

> **Note:** Uncomment the install cell in Setup to enable this section.

---
<a id='part-15'></a>
## Part 15 — Re-ranking

Two re-ranking approaches:
1. **Reciprocal Rank Fusion** (no extra API key — already shown in Part 6)
2. **Cohere Re-Rank** (optional, requires a Cohere API key)

In [52]:
# ── Re-ranking with RRF (Ensuring retriever is defined) ────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from operator import itemgetter
import bs4
import json

# 1. Re-initialize the shared index (required after kernel restarts)
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header"))),
)
blog_docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=300, chunk_overlap=50)
splits = text_splitter.split_documents(blog_docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

# 2. Setup RAG Fusion / RRF logic
template = """You are a helpful assistant that generates multiple search queries based on a single input query.
Generate multiple search queries related to: {question}
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

generate_queries_rrf = (
    prompt_rag_fusion
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

# RRF function already defined in Part 6, but ensuring availability:
def reciprocal_rank_fusion(results: list[list], k: int = 60):
    fused_scores = {}
    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = json.dumps(doc.model_dump())
            fused_scores[doc_str] = fused_scores.get(doc_str, 0) + 1 / (rank + k)
    return [
        (Document.model_validate_json(doc_str), score)
        for doc_str, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

from langchain_core.documents import Document

question = "What is task decomposition for LLM agents?"
retrieval_chain_rrf = generate_queries_rrf | retriever.map() | reciprocal_rank_fusion

# 3. Final Answer Generation
template_ans = """Answer the following question based on this context:
{context}
Question: {question}
"""
prompt_ans = ChatPromptTemplate.from_template(template_ans)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

final_rag_chain_rrf = (
    {"context": retrieval_chain_rrf, "question": itemgetter("question")}
    | prompt_ans
    | llm
    | StrOutputParser()
)

final_rag_chain_rrf.invoke({"question": question})

'Task decomposition for LLM (large language model) agents involves breaking down complex tasks into smaller, manageable subgoals. This process enables the agent to handle complicated tasks more efficiently. There are several methods for task decomposition:\n\n1. **Simple Prompting**: Using straightforward prompts like "Steps for XYZ" or "What are the subgoals for achieving XYZ?" to guide the LLM in identifying the necessary steps.\n\n2. **Task-Specific Instructions**: Providing specific instructions tailored to the task, such as "Write a story outline" for writing a novel.\n\n3. **Human Inputs**: Involving human guidance to help define the subgoals and steps required for the task.\n\nAdditionally, techniques like Chain of Thought (CoT) and Tree of Thoughts (ToT) can be employed. CoT encourages the model to think step by step, while ToT explores multiple reasoning possibilities at each step, creating a tree structure of thoughts. This structured approach aids in clarifying the model\'s 

In [53]:
# -- Cohere Re-Rank (Paste your key below) --
# Requires: pip install cohere langchain_cohere

COHERE_API_KEY = "wgoZVGCOvds9K7yAoweytBzZy6QpAuj5N7dPsuNG" # <--- PASTE YOUR COHERE API KEY HERE

if COHERE_API_KEY:
    try:
        from langchain_cohere import CohereRerank
        from langchain.retrievers import ContextualCompressionRetriever

        # Use k=10 for a wider initial pool to re-rank
        retriever_wide = vectorstore.as_retriever(search_kwargs={"k": 10})

        compressor = CohereRerank(model="rerank-english-v3.0", cohere_api_key=COHERE_API_KEY)
        compression_retriever = ContextualCompressionRetriever(
            base_compressor=compressor, base_retriever=retriever_wide
        )

        question = "What is task decomposition for LLM agents?"
        compressed_docs = compression_retriever.invoke(question)

        print(f"Cohere re-ranked docs: {len(compressed_docs)}")
        for i, d in enumerate(compressed_docs[:3]):
            print(f"\nDoc {i+1} (Relevance Score: {d.metadata.get('relevance_score')}):\n{d.page_content[:200]}...")

    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print("Please provide a COHERE_API_KEY to enable this section.")

Cohere re-ranked docs: 3

Doc 1 (Relevance Score: 0.998844):
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a s...

Doc 2 (Relevance Score: 0.998844):
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a s...

Doc 3 (Relevance Score: 0.998844):
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a s...


In [ ]:
# Re-installing with explicit versions to resolve dependency conflicts caused by langchain-cohere
!pip install -qU --no-cache-dir langchain==0.3.0 langchain-community==0.3.0 langchain-core==0.3.0 langchain-openai langchain-cohere

---
<a id='part-16-18'></a>
## Parts 16–18 — CRAG, Self-RAG, Long Context *(references)*

These sections rely on **LangGraph** for stateful, multi-step agentic workflows.

### Part 16 — Corrective RAG (CRAG)
Evaluates retrieved documents and, if they are irrelevant, falls back to web search before generating.
- Deep dive: https://www.youtube.com/watch?v=E2shqsYwxck
- Notebooks:
  - https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_crag.ipynb
  - https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_crag_mistral.ipynb

### Part 17 — Self-RAG
The model decides when to retrieve, evaluates retrieved passages, and critiques its own generation.
- Notebooks:
  - https://github.com/langchain-ai/langgraph/tree/main/examples/rag
  - https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_self_rag_mistral_nomic.ipynb

### Part 18 — Impact of Long Context
Analysis of how context window length affects RAG performance ("lost in the middle" effect).
- Deep dive: https://www.youtube.com/watch?v=SsHUNfhF32s
- Slides: https://docs.google.com/presentation/d/1mJUiPBdtf58NfuSEQ7pVSEQ2Oqmek7F1i4gBwR6JDss/edit